# Fixed-node architecture comparison

Compares mean, fully connected, MPNN, and GT. This notebook reads small CSV/JSON files only. Every run uses its best available epoch (maximum window-level macro F1), whether or not training has finished.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'results').is_dir(): ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from analysis.architecture_comparison import ARCHITECTURES, COUNTS, load_best_epochs, load_confusions
from io import BytesIO

RESULTS = ROOT / 'results'
df = load_best_epochs(RESULTS)
payloads = load_confusions(RESULTS)
print(f'{len(df)} started runs; {int(df.complete.sum()) if len(df) else 0} finished')
display(df[['architecture','model','n_embeddings','best_epoch','epochs_available','complete','window_macro_f1']])

## Architectures at similar embedding counts

In [ ]:
def plot_within_count(frame, metric='window_macro_f1'):
    fig, axes = plt.subplots(1, 3, figsize=(19, 6), sharey=True)
    colors = dict(zip(ARCHITECTURES, ['#555555','#F58518','#4C78A8','#54A24B']))
    for ax, n in zip(axes, COUNTS):
        part = frame[frame.n_embeddings == n].sort_values(metric, ascending=False)
        if part.empty:
            ax.text(.5,.5,'No epochs available',ha='center',va='center'); ax.set_axis_off(); continue
        ax.barh(part.model, part[metric], color=[colors[a] for a in part.architecture])
        ax.invert_yaxis(); ax.set_title(f'{n} embeddings'); ax.set_xlabel(metric.replace('_',' ')); ax.grid(axis='x',alpha=.25)
        for y, value in enumerate(part[metric]): ax.text(value+.003,y,f'{value:.3f}',va='center',fontsize=8)
    fig.suptitle('Best available epoch for each run'); fig.tight_layout(); return fig
plot_within_count(df);

## Same architecture and feature configuration across embedding counts

In [ ]:
def plot_across_counts(frame, metric='window_macro_f1'):
    fig, axes = plt.subplots(1,4,figsize=(15,4),sharex=True,sharey=True)
    for ax, architecture in zip(axes.flat, ARCHITECTURES):
        part = frame[frame.architecture == architecture]
        for model, series in part.groupby('model'):
            series = series.sort_values('n_embeddings'); ax.plot(series.n_embeddings,series[metric],marker='o',label=model)
        ax.set_title(architecture); ax.set_xticks(COUNTS); ax.grid(alpha=.25); ax.set_xlabel('Number of embeddings'); ax.set_ylabel(metric.replace('_',' '))
        if not part.empty: ax.legend(fontsize=8)
    fig.suptitle('Matched architecture/configuration across embedding counts'); fig.tight_layout(); return fig
plot_across_counts(df);

## Confusion matrix selector

Confusion matrices appear after a run writes its final result JSON. The comparisons above also include incomplete runs. Window level is the default.

In [ ]:
def confusion_png(run, scope='window', normalize=True):
    from io import BytesIO
    data = payloads[run]; field = 'window_test_metrics' if scope == 'window' else 'test_metrics'
    cm = np.asarray(data[field]['confusion_matrix'],dtype=float); classes = data['classes']
    if normalize:
        denominator = cm.sum(axis=1,keepdims=True); cm = np.divide(cm,denominator,out=np.zeros_like(cm),where=denominator!=0)
    fig, ax = plt.subplots(figsize=(9,8)); image = ax.imshow(cm,cmap='Blues',vmin=0,vmax=1 if normalize else None)
    ax.set_xticks(range(len(classes)),classes,rotation=45,ha='right'); ax.set_yticks(range(len(classes)),classes)
    ax.set(xlabel='Predicted class',ylabel='True class',title=f'{run} - {scope} level')
    threshold = 0.5 if normalize else (cm.max() / 2 if cm.size else 0)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            label = f'{cm[i,j]:.2f}' if normalize else f'{int(cm[i,j]):,}'
            ax.text(j,i,label,ha='center',va='center',fontsize=8,color='white' if cm[i,j] > threshold else 'black')
    fig.colorbar(image,ax=ax); fig.tight_layout()
    buffer = BytesIO(); fig.savefig(buffer,format='png',dpi=120,bbox_inches='tight'); plt.close(fig)
    return buffer.getvalue()
if payloads:
    try:
        import ipywidgets as widgets
        from IPython.display import Image, display
        run = widgets.Dropdown(options=sorted(payloads),description='Run:',layout=widgets.Layout(width='750px'))
        scope = widgets.ToggleButtons(options=['window','cell'],description='Level:'); normalize = widgets.Checkbox(value=True,description='Row normalize')
        output = widgets.Output()
        def redraw(change=None):
            png = confusion_png(run.value,scope.value,normalize.value)
            output.clear_output(wait=True)
            with output:
                display(Image(data=png))
        for control in (run,scope,normalize): control.observe(redraw,names='value')
        display(widgets.VBox([run,scope,normalize]),output); redraw()
    except ImportError: print('ipywidgets unavailable; use confusion_png directly')
else: print('No completed result JSONs with confusion matrices yet.')